# OpenPOPCON MANTA example

MANTA is a negative triangularity fusion pilot plant design: 4.55 m, 11 T,
10 MA, delta = -0.448. See Rutherford et al., "MANTA: a
negative-triangularity NASEM-compliant fusion pilot plant",
[Plasma Phys. Control. Fusion 66 (2024) 105006](https://iopscience.iop.org/article/10.1088/1361-6587/ad6708).

Unlike most of the examples, this one reads its geometry and current profile
from a gEQDSK (`gMANTA`) and its density and temperature profiles from a file
(`profsMANTA.csv`).

In [ ]:
import numpy as np
import openpopcon as op

## Setup and run

`settingsfile` holds the machine and algorithm settings, `plotsettingsfile`
the contour levels and axes. Both are read from this directory, and the
gEQDSK and profiles file are found relative to the settings file.

The numerics are compiled with numba the first time they run, so the first
solve in a fresh kernel takes a few seconds longer than the rest.

In [ ]:
settingsfile = "./POPCON_input_example.yaml"
plotsettingsfile = "./plotsettings.yml"

pc = op.POPCON(settingsfile=settingsfile, plotsettingsfile=plotsettingsfile)
pc.run_POPCON()

## Plotting

This example's plotsettings draw the volume-averaged density on the y axis
rather than the Greenwald fraction, and H89 rather than H98.

In [ ]:
fig, ax = pc.plot()

### Adding a custom contour

`custom_plot` draws any (Nn, NTi) array on top of an existing POPCON. Here
it is the bremsstrahlung power, computed point by point from the solver's
own radial model.

In [ ]:
P_brems = np.empty((pc.settings.Nn, pc.settings.NTi))
rho = pc.algorithms.sqrtpsin
for i in range(pc.settings.Nn):
    for j in range(pc.settings.NTi):
        P_brems[i, j] = pc.algorithms.volume_integral(
            rho,
            pc.algorithms._P_brem_rad(
                rho, float(pc.output.T_e_max[j]), float(pc.output.n_e_20_max[i])
            ),
        )

fig, ax = pc.plot(show=False)
pc.custom_plot(
    fig,
    ax,
    P_brems,
    np.linspace(np.min(P_brems), np.max(P_brems), 5),
    "teal",
    1.0,
    r"$P_{brems}$",
)
ax.legend(bbox_to_anchor=(1, 1), loc="upper left")
fig

### A point on the plot

The numbers behind one grid point, picked by Greenwald fraction and
volume-averaged ion temperature.

In [ ]:
i = int(np.abs(pc.output.n_G_frac.values - 0.88).argmin())
j = int(np.abs(pc.output.T_i_avg.values - 7.3).argmin())
point = pc.output.isel(n_index=i, T_index=j)

print(f"n/n_G   = {float(point.n_G_frac):.2f}")
print(f"<T_i>   = {float(point.T_i_avg):.1f} keV")
print(f"P_fus   = {float(point.Pfusion):.1f} MW")
print(f"P_aux   = {float(point.Paux):.1f} MW")
print(f"Q       = {float(point.Q):.2f}")
print(f"beta_N  = {float(point.betaN):.2f}")

## Scoping a single operating point

`single_point` solves one density/temperature pair and shows the profiles
behind it, which is the quickest way to see why a point on the POPCON sits
where it does.

In [ ]:
pc.single_point(n_G_frac=0.88, Ti_av=7.3)

## Scanning the field and the confinement factor

Only parameters that are re-derived on every run can be scanned; the list is
in `openpopcon.SCANNABLE_SETTINGS_KEYS`.

This example supplies a gEQDSK and sets `gfile_rescale: True`, so the
equilibrium supplies the shape (`kappa`, `delta`) and the profiles, and is
moved, resized and given the current of each cell's `R`, `a` and `I_P`.
That keeps `R`, `a`, `I_P` and `qstar` scannable. Flux surface volumes
and areas follow exactly; q and the trapped fraction are rescaled to leading
order in the inverse aspect ratio, so keep geometry scans near the file's own
values. With `gfile_rescale: False` the equilibrium's own geometry and
current are used, and scanning any of those is refused.

The `scan:` block in the settings file varies `B_0` and `H_fac`.

In [ ]:
sc = op.POPCON_scan(settingsfile=settingsfile, plotsettingsfile=plotsettingsfile)
sc.run_scan()

In [ ]:
fig, axs = sc.plot()

`plot_metric` reduces each cell to one number so the trend across the scan
reads at a glance. Anything in the output works, with any reduction.

In [ ]:
fig, ax = sc.plot_metric("Q", reduce="max")

### Scanning the size of the machine

Because `gfile_rescale` is on, the major radius can be scanned with the
equilibrium too. `scale` gives the values relative to the settings file, and
`hold` says what else stays fixed; here the minor radius (the default) and
the current.

In [ ]:
scR = op.POPCON_scan(
    settingsfile=settingsfile,
    plotsettingsfile=plotsettingsfile,
    scan={"rows": {"parameter": "R", "scale": [0.9, 1.0, 1.1], "hold": ["I_P"]}},
)
scR.run_scan()
fig, axs = scR.plot()

In [ ]:
scR.operating_point(T_i_avg=7.3, n_G_frac=0.88)

## Saving and reloading

`write_output` saves a single POPCON or a whole scan, as a directory or a
zip archive, and `read_output` brings it back without re-solving.

In [ ]:
pc.write_output(name="manta", archive=False, overwrite=True)

pcread = op.POPCON()
pcread.read_output("manta")
fig, ax = pcread.plot()

In [ ]:
sc.write_output(name="manta_scan", archive=False, overwrite=True)

back = op.POPCON_scan.read_output("manta_scan")
print(back.shape, back.row.parameter, back.col.parameter)